In [ ]:
import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from PIL import Image



First 100 dataset files moved to: /root/.cache/kagglehub/datasets/d4rklucif3r/cat-and-dogs/versions/1


In [ ]:

# Constants
OUTPUT_CHANNELS = 1
EPOCHS = 2000  # Reduced for testing; increase as needed
BATCH_SIZE = 32
num_examples_to_generate = 16
latent_dim = 100
output_dir = "generated_images2"
checkpoint_dir = './checkpoints'

# Create directories for generated images and checkpoints
os.makedirs(output_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

# Load images function
def load_your_dollar_images():
    images_dir = "dollar_images/"
    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg'))]
    images = []
    for image_file in image_files:
        img_path = os.path.join(images_dir, image_file)
        img = Image.open(img_path).convert('L')  # Convert to grayscale
        img = img.resize((28, 28))  # Resize to 28x28
        img = np.array(img) / 255.0  # Normalize to [0, 1]
        images.append(img)
    return np.array(images)

# Build Generator model
def build_generator():
    model = tf.keras.Sequential([
        layers.Dense(256, use_bias=False, input_shape=(latent_dim,)),
        layers.LeakyReLU(),
        layers.Dense(28 * 28 * OUTPUT_CHANNELS, use_bias=False, activation='tanh'),
        layers.Reshape((28, 28, OUTPUT_CHANNELS))
    ])
    return model

# Build Discriminator model
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Flatten(input_shape=(28, 28, OUTPUT_CHANNELS)),
        layers.Dense(256, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Save generated images in a grid
def save_generated_images(epoch, generator):
    noise = tf.random.normal([num_examples_to_generate, latent_dim])
    generated_images = generator(noise, training=False)
    fig, axs = plt.subplots(4, 4, figsize=(6, 6))  # Create 4x4 grid
    for i, ax in enumerate(axs.flat):
        img = (generated_images[i] + 1) / 2.0  # Rescale to [0, 1]
        ax.imshow(img.numpy().squeeze(), cmap='gray')
        ax.axis('off')
    plt.savefig(os.path.join(output_dir, f"epoch_{epoch}.png"))
    plt.close()

# GAN training
def train_gan():
    generator = build_generator()
    discriminator = build_discriminator()

    generator_optimizer = tf.keras.optimizers.Adam(1e-4)
    discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)
    cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

    # Loss functions
    def discriminator_loss(real_output, fake_output):
        real_loss = cross_entropy(tf.ones_like(real_output), real_output)
        fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
        return real_loss + fake_loss

    def generator_loss(fake_output):
        return cross_entropy(tf.ones_like(fake_output), fake_output)

    # Load dataset
    train_images = load_your_dollar_images()
    train_images = (train_images - 0.5) * 2  # Normalize to [-1, 1]
    train_images = np.expand_dims(train_images, axis=-1)

    # Create a checkpoint manager
    checkpoint = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                                      generator_optimizer=generator_optimizer,
                                      discriminator_optimizer=discriminator_optimizer)
    checkpoint_manager = tf.train.CheckpointManager(checkpoint, checkpoint_dir, max_to_keep=5)

    # Training loop
    for epoch in range(EPOCHS):
        idx = np.random.randint(0, train_images.shape[0], BATCH_SIZE)
        real_images = train_images[idx]
        noise = tf.random.normal([BATCH_SIZE, latent_dim])

        # Train Discriminator
        with tf.GradientTape() as disc_tape:
            real_output = discriminator(real_images, training=True)
            fake_images = generator(noise, training=True)
            fake_output = discriminator(fake_images, training=True)
            disc_loss = discriminator_loss(real_output, fake_output)
        gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
        discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

        # Train Generator
        with tf.GradientTape() as gen_tape:
            fake_images = generator(noise, training=True)
            fake_output = discriminator(fake_images, training=True)
            gen_loss = generator_loss(fake_output)
        gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
        generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))

        # Save generated images and checkpoints periodically
        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch + 1}, Generator Loss: {gen_loss.numpy()}, Discriminator Loss: {disc_loss.numpy()}")
            save_generated_images(epoch + 1, generator)
            checkpoint_manager.save()

# Start training
train_gan()


Epoch 100, Generator Loss: 0.7610237002372742, Discriminator Loss: 0.8527345657348633
Epoch 200, Generator Loss: 0.963488757610321, Discriminator Loss: 0.6724401712417603
Epoch 300, Generator Loss: 1.2819883823394775, Discriminator Loss: 0.5027469396591187
Epoch 400, Generator Loss: 1.637582540512085, Discriminator Loss: 0.36503058671951294
Epoch 500, Generator Loss: 1.9172077178955078, Discriminator Loss: 0.28373727202415466
Epoch 600, Generator Loss: 1.9838042259216309, Discriminator Loss: 0.2650775611400604
Epoch 700, Generator Loss: 1.843125343322754, Discriminator Loss: 0.32338953018188477
Epoch 800, Generator Loss: 1.8542696237564087, Discriminator Loss: 0.3084093928337097
Epoch 900, Generator Loss: 1.6818506717681885, Discriminator Loss: 0.41387006640434265
Epoch 1000, Generator Loss: 2.0439209938049316, Discriminator Loss: 0.3064500689506531
Epoch 1100, Generator Loss: 1.9862914085388184, Discriminator Loss: 0.32544878125190735
Epoch 1200, Generator Loss: 2.190037965774536, Dis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')